# 10 — Runtime access and event-log configuration
The Billing API project preflight requires `resourcemanager.projects.get`. Grant that single read permission to the dedicated VM service account on the authorised project. Keep billing-change permissions absent. Ensure Spark History Server reads the notebook event logs.
Reference: https://docs.cloud.google.com/billing/docs/reference/rest/v1/projects/getBillingInfo

In [ ]:
from pathlib import Path
import json, re, shlex, subprocess
from IPython.display import display
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks": REPO_ROOT = REPO_ROOT.parent
EXPECTED_RUN_ID = "20261012-full-01"
runtime = json.loads(
    (REPO_ROOT / "artifacts/execution/runs" / EXPECTED_RUN_ID / "runtime.json").read_text()
)
PROJECT_ID, BUCKET = runtime["project_id"], runtime["bucket"]
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
from rainfall.billing import validate_runtime_credit_gate
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unapproved billing/credit.")
def sdk(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr))
    return result.stdout
billing = json.loads(sdk(["billing", "projects", "describe", PROJECT_ID, "--format=json"]))
display(validate_runtime_credit_gate(runtime, project_id=PROJECT_ID, billing_info=billing))
cluster = json.loads(sdk(["dataproc", "clusters", "describe", "dsa5208-rainfall", f"--project={PROJECT_ID}", "--region=asia-southeast1", "--format=json"]))
zone = cluster["config"]["gceClusterConfig"]["zoneUri"].rsplit("/",1)[-1]
master = cluster["config"]["masterConfig"]["instanceNames"][0]
def ssh(command):
    return sdk(["compute", "ssh", master, f"--project={PROJECT_ID}", f"--zone={zone}", "--tunnel-through-iap", "--quiet", f"--command={command}"])


In [2]:
ROLE_ID = "rainfallPreflightReader"
role = subprocess.run(["gcloud", "iam", "roles", "describe", ROLE_ID, f"--project={PROJECT_ID}", "--format=json"], capture_output=True, text=True)
if role.returncode:
    if "NOT_FOUND" not in role.stderr and "does not exist" not in role.stderr:
        raise RuntimeError("Cannot inspect the custom role; do not change IAM blindly.")
    sdk(["iam", "roles", "create", ROLE_ID, f"--project={PROJECT_ID}", "--title=Rainfall preflight read", "--permissions=resourcemanager.projects.get", "--stage=GA", "--format=none"])
else:
    if json.loads(role.stdout).get("includedPermissions") != ["resourcemanager.projects.get"]:
        raise RuntimeError("Existing role has unexpected permissions.")
sdk(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=serviceAccount:rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com", f"--role=projects/{PROJECT_ID}/roles/{ROLE_ID}", "--condition=None", "--format=none"])
print("Granted only resourcemanager.projects.get on the authorised project; no billing-change permission.")
marker = REPO_ROOT / "artifacts/execution/spark-history-marker.txt"
marker.write_text("Spark event-log directory marker.\n")
sdk(["storage", "cp", str(marker), f"gs://{BUCKET}/spark-history/.keep"])


Granted only resourcemanager.projects.get on the authorised project; no billing-change permission.


''

In [3]:
inspect_code = """from pathlib import Path
import json, subprocess
config = Path('/etc/spark/conf/spark-defaults.conf').read_text()
lines = [line for line in config.splitlines() if line.strip().startswith(('spark.eventLog.dir', 'spark.history.fs.logDirectory'))]
units = subprocess.check_output(['systemctl', 'list-units', '--type=service', '--all', '--no-legend', '--no-pager'], text=True)
units = [line.strip() for line in units.splitlines() if 'spark-history' in line or 'knox' in line]
print(json.dumps({'configuration':lines,'services':units}))
"""
info = json.loads(ssh("python3 -c " + shlex.quote(inspect_code)))
display(info)


{'configuration': ['spark.eventLog.dir=gs://sg-rainfall-dsa5208-261011-data/80a97e6e-f4e3-4804-b0cd-b2cba517ed9f/spark-job-history',
  'spark.history.fs.logDirectory=gs://sg-rainfall-dsa5208-261011-data/80a97e6e-f4e3-4804-b0cd-b2cba517ed9f/spark-job-history',
  'spark.eventLog.dir=gs\\://sg-rainfall-dsa5208-261011-data/spark-history/'],
 'services': ['knox.service                              loaded    active     running         Knox gateway service',
  'spark-history-server.service              loaded    active     running         LSB: Spark history-server']}

In [4]:
# Point the history viewer at the same event-log directory as notebook applications.
history_units = [line.split()[0] for line in info["services"] if "spark-history" in line]
if len(history_units) != 1:
    raise RuntimeError("Expected exactly one Spark History Server service.")
unit = history_units[0]
edit_code = """from pathlib import Path
p = Path('/etc/spark/conf/spark-defaults.conf')
text = p.read_text()
lines = [line for line in text.splitlines() if not line.strip().startswith('spark.history.fs.logDirectory')]
lines.append('spark.history.fs.logDirectory gs://BUCKET_PLACEHOLDER/spark-history/')
p.write_text(chr(10).join(lines) + chr(10))
""".replace("BUCKET_PLACEHOLDER", BUCKET)
print(ssh("sudo python3 -c " + shlex.quote(edit_code)))
print(ssh("sudo systemctl restart " + shlex.quote(unit)))
print(ssh("systemctl show " + shlex.quote(unit) + " --property=ActiveState,SubState,Result,ExecMainStatus"))
print(ssh("gcloud billing projects describe " + shlex.quote(PROJECT_ID) + " --format=json"))


Result=success
ExecMainStatus=0
ActiveState=active
SubState=running



{
  "billingAccountName": "billingAccounts/01DAD8-AE5F1A-E3910A",
  "billingEnabled": true,
  "name": "projects/sg-rainfall-dsa5208-261011/billingInfo",
  "projectId": "sg-rainfall-dsa5208-261011"
}

